# Airline Web Scraper Testing Notebook
## Test Emirates, Airblue, and PIA Flight Scrapers

This notebook tests web scraping functionality for three Pakistani airlines:
- **Emirates** (from Pakistan origins)
- **Airblue** 
- **PIA (Pakistan International Airlines)**

**Current Issue:** Emirates scraper is failing due to cookie consent popup blocking element clicks.

### Instructions:
1. Run cells sequentially
2. Choose `headless=False` to see browser actions (useful for debugging)
3. Each airline has a dedicated test section
4. Results will be displayed in pandas DataFrames

## 1. Import Required Libraries

In [ ]:
import selenium
from selenium import webdriver
from selenium.webdriver.common.by import By
from selenium.webdriver.support.ui import WebDriverWait
from selenium.webdriver.support import expected_conditions as EC
from selenium.webdriver.chrome.service import Service
from selenium.webdriver.chrome.options import Options
from selenium.common.exceptions import TimeoutException, NoSuchElementException, ElementClickInterceptedException
from datetime import datetime, timedelta
import pandas as pd
import time
import re

print("✓ All imports successful")

ModuleNotFoundError: No module named 'selenium'

## 2. Configure Chrome WebDriver

**Set `headless=False`** to see the browser window and debug issues visually.

In [ ]:
def setup_driver(headless=False):
    """Configure and return Chrome WebDriver"""
    options = Options()
    
    if headless:
        options.add_argument('--headless')
    
    # Important options for compatibility
    options.add_argument('--no-sandbox')
    options.add_argument('--disable-dev-shm-usage')
    options.add_argument('--disable-gpu')
    options.add_argument('--window-size=1920,1080')
    
    # User agent to appear as regular browser
    options.add_argument('user-agent=Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/120.0.0.0 Safari/537.36')
    
    driver = webdriver.Chrome(options=options)
    driver.set_page_load_timeout(60)
    
    return driver

print("✓ WebDriver configuration ready")

## 3. Helper Functions

In [ ]:
def smart_delay(seconds, variance=0.5):
    """Add random delay to avoid detection"""
    import random
    time.sleep(seconds + random.uniform(-variance, variance))

def handle_cookie_popup(driver, wait):
    """Handle Emirates cookie consent popup"""
    try:
        # Look for cookie accept button
        cookie_selectors = [
            "button[id*='accept']",
            "button[class*='accept']",
            "button.onetrust-close-btn-handler",
            "#onetrust-accept-btn-handler",
            "button.accept-cookies"
        ]
        
        for selector in cookie_selectors:
            try:
                button = wait.until(EC.element_to_be_clickable((By.CSS_SELECTOR, selector)))
                button.click()
                print("✓ Cookie popup dismissed")
                time.sleep(2)
                return True
            except:
                continue
        
        # Try JavaScript click as last resort
        try:
            driver.execute_script("""
                let buttons = document.querySelectorAll('button');
                for (let btn of buttons) {
                    if (btn.textContent.toLowerCase().includes('accept') || 
                        btn.textContent.toLowerCase().includes('agree')) {
                        btn.click();
                        return true;
                    }
                }
            """)
            print("✓ Cookie popup dismissed via JavaScript")
            time.sleep(2)
            return True
        except:
            pass
            
    except Exception as e:
        print(f"⚠ Could not dismiss cookie popup: {e}")
    
    return False

print("✓ Helper functions defined")

## 4. Test Emirates Scraper

**Known Issue:** Cookie consent popup blocking element clicks.

This test will:
1. Load Emirates flight schedule page
2. Handle cookie popup
3. Search for flights from Karachi/Lahore/Islamabad
4. Extract flight information

In [ ]:
def test_emirates_scraper(origin_city="Karachi", headless=False):
    """Test Emirates flight scraper with improved cookie handling"""
    
    print(f"🔍 Testing Emirates scraper for {origin_city}...")
    print("="*60)
    
    driver = None
    try:
        driver = setup_driver(headless=headless)
        wait = WebDriverWait(driver, 20)
        
        # Load Emirates page
        url = "https://www.emirates.com/pk/english/flights-to-dubai/"
        print(f"Loading: {url}")
        driver.get(url)
        time.sleep(3)
        
        # CRITICAL: Handle cookie popup FIRST
        print("\n1. Handling cookie popup...")
        handle_cookie_popup(driver, wait)
        
        # Wait for React app
        print("\n2. Waiting for page to load...")
        wait.until(EC.presence_of_element_located((By.ID, "__next")))
        print("✓ Page loaded")
        
        # Try to find origin input
        print(f"\n3. Searching for flights from {origin_city}...")
        try:
            origin_input = wait.until(EC.element_to_be_clickable((By.ID, "origin-airport")))
            
            # Remove readonly and enter city
            driver.execute_script("arguments[0].removeAttribute('readonly')", origin_input)
            driver.execute_script("arguments[0].click()", origin_input)  # JavaScript click
            time.sleep(1)
            
            origin_input.clear()
            origin_input.send_keys(origin_city)
            print(f"✓ Entered: {origin_city}")
            time.sleep(2)
            
            # Look for autocomplete results
            dropdown = wait.until(EC.presence_of_element_located((By.ID, "origin-airport_list")))
            options = dropdown.find_elements(By.TAG_NAME, "li")
            
            if options:
                print(f"✓ Found {len(options)} autocomplete options")
                options[0].click()
                print("✓ Selected first option")
            
            time.sleep(2)
            
            # Extract flight data (simplified for testing)
            print("\n4. Extracting flight information...")
            page_html = driver.page_source
            
            # Count flight-related elements
            flight_elements = driver.find_elements(By.CLASS_NAME, "flight-details")
            print(f"✓ Found {len(flight_elements)} flight elements")
            
            if len(flight_elements) == 0:
                print("⚠ No flights found - page may need more time or different selectors")
            
            return {
                'status': 'SUCCESS' if len(flight_elements) > 0 else 'PARTIAL',
                'origin': origin_city,
                'flights_found': len(flight_elements),
                'page_title': driver.title
            }
            
        except ElementClickInterceptedException as e:
            print(f"❌ Element click blocked: {e}")
            print("\n🔍 Checking for blocking elements...")
            
            blocking = driver.find_elements(By.CSS_SELECTOR, ".onetrust-pc-dark-filter, .cookie-banner, [class*='overlay']")
            if blocking:
                print(f"Found {len(blocking)} blocking elements")
                for elem in blocking:
                    print(f"  - {elem.tag_name} class={elem.get_attribute('class')}")
            
            return {'status': 'FAILED', 'error': 'Cookie popup blocking clicks', 'origin': origin_city}
            
    except Exception as e:
        print(f"❌ Error: {type(e).__name__}: {e}")
        return {'status': 'ERROR', 'error': str(e), 'origin': origin_city}
        
    finally:
        if driver:
            driver.quit()
            print("\n✓ Browser closed")

# Run test (set headless=False to see browser)
result = test_emirates_scraper(origin_city="Karachi", headless=False)
print("\n" + "="*60)
print("RESULT:", result)

## 5. Test PIA Scraper

PIA (Pakistan International Airlines) flight scraper test.

In [ ]:
def test_pia_scraper(headless=False):
    """Test PIA flight scraper"""
    
    print("🔍 Testing PIA scraper...")
    print("="*60)
    
    driver = None
    try:
        driver = setup_driver(headless=headless)
        wait = WebDriverWait(driver, 20)
        
        # PIA website URL
        url = "https://www.piac.com.pk/"
        print(f"Loading: {url}")
        driver.get(url)
        time.sleep(3)
        
        print(f"✓ Page loaded: {driver.title}")
        
        # Look for flight search or schedule sections
        print("\n🔍 Analyzing page structure...")
        
        # Common PIA selectors
        selectors_to_check = [
            ("Flight Search", "input[placeholder*='Origin'], input[name*='origin']"),
            ("Book Flight", "a[href*='book'], button:contains('Book')"),
            ("Flight Schedule", "a[href*='schedule'], a:contains('Schedule')"),
            ("Search Button", "button[type='submit'], .search-button")
        ]
        
        findings = []
        for name, selector in selectors_to_check:
            try:
                elements = driver.find_elements(By.CSS_SELECTOR, selector)
                if elements:
                    findings.append(f"✓ Found {name}: {len(elements)} elements")
                else:
                    findings.append(f"✗ {name}: Not found")
            except:
                findings.append(f"✗ {name}: Error checking")
        
        for finding in findings:
            print(finding)
        
        # Extract page links
        links = driver.find_elements(By.TAG_NAME, "a")
        relevant_links = [link.get_attribute('href') for link in links 
                         if link.get_attribute('href') and 
                         any(keyword in link.get_attribute('href').lower() 
                             for keyword in ['schedule', 'flight', 'book'])]
        
        print(f"\n✓ Found {len(relevant_links)} relevant links")
        for link in relevant_links[:5]:
            print(f"  - {link}")
        
        return {
            'status': 'ANALYZED',
            'page_title': driver.title,
            'relevant_links': len(relevant_links),
            'findings': findings
        }
        
    except Exception as e:
        print(f"❌ Error: {e}")
        return {'status': 'ERROR', 'error': str(e)}
        
    finally:
        if driver:
            driver.quit()
            print("\n✓ Browser closed")

# Run PIA test
result = test_pia_scraper(headless=False)
print("\n" + "="*60)
print("RESULT:", result)

## 6. Test Airblue Scraper

Airblue flight scraper test.

In [ ]:
def test_airblue_scraper(headless=False):
    """Test Airblue flight scraper"""
    
    print("🔍 Testing Airblue scraper...")
    print("="*60)
    
    driver = None
    try:
        driver = setup_driver(headless=headless)
        wait = WebDriverWait(driver, 20)
        
        # Airblue website
        url = "https://www.airblue.com/"
        print(f"Loading: {url}")
        driver.get(url)
        time.sleep(3)
        
        print(f"✓ Page loaded: {driver.title}")
        
        # Look for booking/schedule sections
        print("\n🔍 Analyzing page structure...")
        
        selectors_to_check = [
            ("Origin Input", "input[placeholder*='From'], input[name*='origin']"),
            ("Destination Input", "input[placeholder*='To'], input[name*='destination']"),
            ("Date Picker", "input[type='date'], .date-picker"),
            ("Search/Book Button", "button:contains('Search'), button:contains('Book')"),
            ("Flight Info", ".flight-info, .flight-details")
        ]
        
        findings = []
        for name, selector in selectors_to_check:
            try:
                elements = driver.find_elements(By.CSS_SELECTOR, selector)
                if elements:
                    findings.append(f"✓ Found {name}: {len(elements)} elements")
                else:
                    findings.append(f"✗ {name}: Not found")
            except:
                findings.append(f"✗ {name}: Error checking")
        
        for finding in findings:
            print(finding)
        
        # Extract navigation links
        nav_links = driver.find_elements(By.CSS_SELECTOR, "nav a, .menu a")
        print(f"\n✓ Found {len(nav_links)} navigation links")
        
        for link in nav_links[:10]:
            text = link.text.strip()
            href = link.get_attribute('href')
            if text and href:
                print(f"  - {text}: {href}")
        
        return {
            'status': 'ANALYZED',
            'page_title': driver.title,
            'nav_links': len(nav_links),
            'findings': findings
        }
        
    except Exception as e:
        print(f"❌ Error: {e}")
        return {'status': 'ERROR', 'error': str(e)}
        
    finally:
        if driver:
            driver.quit()
            print("\n✓ Browser closed")

# Run Airblue test
result = test_airblue_scraper(headless=False)
print("\n" + "="*60)
print("RESULT:", result)

## 7. Compare All Scrapers

Run all three scrapers and display comparison results.

In [ ]:
def run_all_scraper_tests(headless=True):
    """Run all scraper tests and compare results"""
    
    results = []
    
    # Test Emirates
    print("\n" + "🛫 EMIRATES TEST ".center(60, "="))
    emirates_result = test_emirates_scraper(origin_city="Karachi", headless=headless)
    results.append({'Airline': 'Emirates', **emirates_result})
    
    print("\n\n" + "🛫 PIA TEST ".center(60, "="))
    pia_result = test_pia_scraper(headless=headless)
    results.append({'Airline': 'PIA', **pia_result})
    
    print("\n\n" + "🛫 AIRBLUE TEST ".center(60, "="))
    airblue_result = test_airblue_scraper(headless=headless)
    results.append({'Airline': 'Airblue', **airblue_result})
    
    # Create DataFrame
    df = pd.DataFrame(results)
    
    print("\n\n" + "📊 SCRAPER TEST SUMMARY ".center(60, "="))
    print(df.to_string(index=False))
    
    return df

# Run all tests (set headless=True to run faster without browser window)
results_df = run_all_scraper_tests(headless=False)
results_df

## 8. Fix Emirates Cookie Popup Issue

Based on test results, implement cookie popup fix.

### Findings & Recommendations:

**Emirates Issue:** 
- Cookie consent popup (`onetrust-pc-dark-filter`) with z-index 2147483645 blocks all clicks
- Need to dismiss popup before interacting with page elements

**Solution:** Add `handle_cookie_popup()` function to Java code:
```java
// Add after page load in EmiratesFlightScraperService.java line 255
driver.executeScript(
    "document.querySelectorAll('.onetrust-pc-dark-filter, #onetrust-banner-sdk').forEach(e => e.remove());" +
    "let buttons = document.querySelectorAll('button');" +
    "for (let btn of buttons) {" +
    "    if (btn.textContent.toLowerCase().includes('accept')) {" +
    "        btn.click();" +
    "        break;" +
    "    }" +
    "}"
);
Thread.sleep(2000);
```

**PIA & Airblue:**
- Need to analyze their current scraper implementations in the Java code
- May need different selectors or API endpoints